<a href="https://colab.research.google.com/github/AshokGit544/Utility-Data-Pipeline-Orchestration-Airflow/blob/main/Utility_Data_Pipeline_Orchestration_Airflow.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from pathlib import Path
from datetime import datetime, timedelta
import random
import json
import time

import pandas as pd
import numpy as np

BASE_PATH = Path('/content/utility_data_pipeline_orchestration_airflow')
RAW_PATH = BASE_PATH / 'data' / 'raw'
BRONZE_PATH = BASE_PATH / 'data' / 'bronze'
SILVER_PATH = BASE_PATH / 'data' / 'silver'
GOLD_PATH = BASE_PATH / 'data' / 'gold'
OUTPUT_PATH = BASE_PATH / 'outputs'

for p in [RAW_PATH, BRONZE_PATH, SILVER_PATH, GOLD_PATH, OUTPUT_PATH]:
    p.mkdir(parents=True, exist_ok=True)

print('Project folders created successfully.')
print(BASE_PATH)

Project folders created successfully.
/content/utility_data_pipeline_orchestration_airflow


In [ ]:
random.seed(42)
np.random.seed(42)

num_customers = 1000
num_meter_records = 15000
num_outages = 1200
num_billing = 3000
regions = ['North', 'South', 'East', 'West', 'Central']
outage_causes = ['Storm', 'Equipment Failure', 'Vegetation', 'Animal Contact', 'Unknown']
start_date = datetime(2024, 1, 1)

# customers
customers = []
for i in range(1, num_customers + 1):
    customers.append({
        'customer_id': f'CUST{i:05d}',
        'region': random.choice(regions),
        'customer_type': random.choice(['Residential', 'Commercial', 'Industrial']),
        'status': random.choice(['Active', 'Active', 'Active', 'Inactive'])
    })
customers_df = pd.DataFrame(customers)
customers_df.to_csv(RAW_PATH / 'customers.csv', index=False)

# meter usage
meter = []
for _ in range(num_meter_records):
    cust = random.randint(1, num_customers)
    reading_date = start_date + timedelta(days=random.randint(0, 179))
    meter.append({
        'customer_id': f'CUST{cust:05d}',
        'reading_date': reading_date.strftime('%Y-%m-%d'),
        'kwh_usage': round(float(max(5, np.random.normal(35, 10))), 2),
        'estimated_read_flag': random.choice([0, 0, 0, 1])
    })
meter_df = pd.DataFrame(meter)
meter_df.loc[5, 'kwh_usage'] = None
meter_df = pd.concat([meter_df, meter_df.iloc[:20]], ignore_index=True)
meter_df.to_csv(RAW_PATH / 'meter_usage.csv', index=False)

# outages
outages = []
for i in range(1, num_outages + 1):
    cust = random.randint(1, num_customers)
    outage_date = start_date + timedelta(days=random.randint(0, 179))
    outages.append({
        'outage_id': f'OUT{i:06d}',
        'customer_id': f'CUST{cust:05d}',
        'outage_date': outage_date.strftime('%Y-%m-%d'),
        'duration_minutes': max(5, int(np.random.normal(80, 30))),
        'cause': random.choice(outage_causes)
    })
outages_df = pd.DataFrame(outages)
outages_df.to_csv(RAW_PATH / 'outages.csv', index=False)

# billing
billing = []
for i in range(1, num_billing + 1):
    cust = random.randint(1, num_customers)
    bill_date = start_date + timedelta(days=random.randint(0, 179))
    billing.append({
        'billing_id': f'BILL{i:06d}',
        'customer_id': f'CUST{cust:05d}',
        'bill_date': bill_date.strftime('%Y-%m-%d'),
        'bill_amount': round(float(max(40, np.random.normal(145, 40))), 2),
        'payment_status': random.choice(['Paid', 'Paid', 'Pending', 'Overdue'])
    })
billing_df = pd.DataFrame(billing)
billing_df.to_csv(RAW_PATH / 'billing.csv', index=False)

print('Raw files created:')
for file in RAW_PATH.glob('*.csv'):
    print(file.name)


Raw files created:
outages.csv
meter_usage.csv
billing.csv
customers.csv


In [1]:
execution_logs = []

def log_task(task_name, status, start_time, end_time, message=''):
    execution_logs.append({
        'task_name': task_name,
        'status': status,
        'start_time': start_time.isoformat(),
        'end_time': end_time.isoformat(),
        'duration_seconds': round((end_time - start_time).total_seconds(), 2),
        'message': message
    })

def run_task(task_name, task_function, retries=1, retry_delay=2):
    attempt = 0
    while attempt <= retries:
        start_time = datetime.now()
        try:
            result = task_function()
            end_time = datetime.now()
            log_task(task_name, 'SUCCESS', start_time, end_time, f'Completed on attempt {attempt + 1}')
            print(f'{task_name}: SUCCESS')
            return result
        except Exception as e:
            end_time = datetime.now()
            attempt += 1
            if attempt <= retries:
                log_task(task_name, 'RETRY', start_time, end_time, f'Attempt {attempt} failed: {str(e)}')
                print(f'{task_name}: RETRYING because {e}')
                time.sleep(retry_delay)
            else:
                log_task(task_name, 'FAILED', start_time, end_time, str(e))
                print(f'{task_name}: FAILED')
                raise

In [ ]:
def validate_raw_data():
    required_files = ['customers.csv', 'meter_usage.csv', 'outages.csv', 'billing.csv']
    missing = [f for f in required_files if not (RAW_PATH / f).exists()]

    if missing:
        raise FileNotFoundError(f'Missing raw files: {missing}')

    customers_df = pd.read_csv(RAW_PATH / 'customers.csv')
    meter_df = pd.read_csv(RAW_PATH / 'meter_usage.csv')
    outages_df = pd.read_csv(RAW_PATH / 'outages.csv')
    billing_df = pd.read_csv(RAW_PATH / 'billing.csv')

    if customers_df.empty:
        raise ValueError('customers.csv is empty')
    if meter_df.empty:
        raise ValueError('meter_usage.csv is empty')
    if outages_df.empty:
        raise ValueError('outages.csv is empty')
    if billing_df.empty:
        raise ValueError('billing.csv is empty')

    print("Raw validation passed")
    return "Raw validation completed"


def load_bronze_layer():
    for file in RAW_PATH.glob('*.csv'):
        df = pd.read_csv(file)
        df.to_csv(BRONZE_PATH / file.name, index=False)

    print("Bronze layer loaded")
    return "Bronze layer loaded"


def transform_silver_layer():
    customers_df = pd.read_csv(BRONZE_PATH / 'customers.csv')
    meter_df = pd.read_csv(BRONZE_PATH / 'meter_usage.csv')
    outages_df = pd.read_csv(BRONZE_PATH / 'outages.csv')
    billing_df = pd.read_csv(BRONZE_PATH / 'billing.csv')

    customers_df = customers_df.drop_duplicates(subset=['customer_id'])

    meter_df['reading_date'] = pd.to_datetime(meter_df['reading_date'])
    meter_df['kwh_usage'] = meter_df['kwh_usage'].fillna(0)
    meter_df = meter_df[meter_df['kwh_usage'] >= 0]
    meter_df = meter_df.drop_duplicates(subset=['customer_id', 'reading_date'])

    outages_df['outage_date'] = pd.to_datetime(outages_df['outage_date'])
    outages_df['duration_minutes'] = outages_df['duration_minutes'].fillna(0)
    outages_df = outages_df[outages_df['duration_minutes'] >= 0]

    billing_df['bill_date'] = pd.to_datetime(billing_df['bill_date'])
    billing_df['bill_amount'] = billing_df['bill_amount'].fillna(0)
    billing_df = billing_df[billing_df['bill_amount'] >= 0]

    customers_df.to_csv(SILVER_PATH / 'customers.csv', index=False)
    meter_df.to_csv(SILVER_PATH / 'meter_usage.csv', index=False)
    outages_df.to_csv(SILVER_PATH / 'outages.csv', index=False)
    billing_df.to_csv(SILVER_PATH / 'billing.csv', index=False)

    print("Silver layer transformed")
    return "Silver layer transformed"


def publish_gold_layer():
    customers_df = pd.read_csv(SILVER_PATH / 'customers.csv')
    meter_df = pd.read_csv(SILVER_PATH / 'meter_usage.csv')
    outages_df = pd.read_csv(SILVER_PATH / 'outages.csv')
    billing_df = pd.read_csv(SILVER_PATH / 'billing.csv')

    meter_daily = meter_df.groupby(['customer_id', 'reading_date'], as_index=False).agg(
        daily_kwh_usage=('kwh_usage', 'sum'),
        has_estimated_read=('estimated_read_flag', 'max')
    )
    meter_daily = meter_daily.rename(columns={'reading_date': 'event_date'})

    outage_daily = outages_df.groupby(['customer_id', 'outage_date'], as_index=False).agg(
        daily_outage_count=('outage_id', 'count'),
        daily_outage_minutes=('duration_minutes', 'sum')
    )
    outage_daily = outage_daily.rename(columns={'outage_date': 'event_date'})

    billing_daily = billing_df.groupby(['customer_id', 'bill_date'], as_index=False).agg(
        daily_billed_amount=('bill_amount', 'sum'),
        daily_bill_count=('billing_id', 'count')
    )
    billing_daily = billing_daily.rename(columns={'bill_date': 'event_date'})

    gold_df = meter_daily.merge(customers_df, on='customer_id', how='left')
    gold_df = gold_df.merge(outage_daily, on=['customer_id', 'event_date'], how='left')
    gold_df = gold_df.merge(billing_daily, on=['customer_id', 'event_date'], how='left')

    gold_df['daily_outage_count'] = gold_df['daily_outage_count'].fillna(0)
    gold_df['daily_outage_minutes'] = gold_df['daily_outage_minutes'].fillna(0)
    gold_df['daily_billed_amount'] = gold_df['daily_billed_amount'].fillna(0)
    gold_df['daily_bill_count'] = gold_df['daily_bill_count'].fillna(0)

    gold_df.to_csv(GOLD_PATH / 'enterprise_gold.csv', index=False)

    print("Gold layer published")
    return "Gold layer published"


def generate_kpis():
    gold_df = pd.read_csv(GOLD_PATH / 'enterprise_gold.csv')

    kpi_df = gold_df.groupby('region', as_index=False).agg(
        avg_daily_kwh_usage=('daily_kwh_usage', 'mean'),
        avg_daily_outage_minutes=('daily_outage_minutes', 'mean'),
        total_billed_amount=('daily_billed_amount', 'sum')
    )

    kpi_df.to_csv(OUTPUT_PATH / 'kpi_output.csv', index=False)

    print("KPI output generated")
    return "KPI output generated"


def write_pipeline_summary():
    summary = {
        'generated_at': datetime.now().isoformat(),
        'raw_files': len(list(RAW_PATH.glob('*.csv'))),
        'bronze_files': len(list(BRONZE_PATH.glob('*.csv'))),
        'silver_files': len(list(SILVER_PATH.glob('*.csv'))),
        'gold_files': len(list(GOLD_PATH.glob('*.csv'))),
        'output_files': len(list(OUTPUT_PATH.glob('*')))
    }

    with open(OUTPUT_PATH / 'pipeline_summary.json', 'w') as f:
        json.dump(summary, f, indent=2)

    print("Pipeline summary written")
    return summary

In [ ]:
run_task('validate_raw_data', validate_raw_data, retries=1, retry_delay=2)
run_task('load_bronze_layer', load_bronze_layer, retries=1, retry_delay=2)
run_task('transform_silver_layer', transform_silver_layer, retries=1, retry_delay=2)
run_task('publish_gold_layer', publish_gold_layer, retries=1, retry_delay=2)
run_task('generate_kpis', generate_kpis, retries=1, retry_delay=2)
summary_result = run_task('write_pipeline_summary', write_pipeline_summary, retries=0)

print('Pipeline completed successfully.')
print(summary_result)

Raw validation passed
validate_raw_data: SUCCESS
Bronze layer loaded
load_bronze_layer: SUCCESS
Silver layer transformed
transform_silver_layer: SUCCESS
Gold layer published
publish_gold_layer: SUCCESS
KPI output generated
generate_kpis: SUCCESS
Pipeline summary written
write_pipeline_summary: SUCCESS
Pipeline completed successfully.
{'generated_at': '2026-03-17T03:45:52.451213', 'raw_files': 4, 'bronze_files': 4, 'silver_files': 4, 'gold_files': 1, 'output_files': 1}


In [ ]:
execution_log_df = pd.DataFrame(execution_logs)
execution_log_df.to_csv(OUTPUT_PATH / 'pipeline_execution_log.csv', index=False)
execution_log_df

,task_name,status,start_time,end_time,duration_seconds,message
0,validate_raw_data,RETRY,2026-03-17T03:41:38.078812,2026-03-17T03:41:38.078828,0.00,Attempt 1 failed: 'list' object has no attribu...
1,validate_raw_data,FAILED,2026-03-17T03:41:40.079101,2026-03-17T03:41:40.079136,0.00,'list' object has no attribute 'to_csv'
2,validate_raw_data,SUCCESS,2026-03-17T03:45:51.973151,2026-03-17T03:45:52.002846,0.03,Completed on attempt 1
3,load_bronze_layer,SUCCESS,2026-03-17T03:45:52.003096,2026-03-17T03:45:52.080263,0.08,Completed on attempt 1
4,transform_silver_layer,SUCCESS,2026-03-17T03:45:52.080473,2026-03-17T03:45:52.200493,0.12,Completed on attempt 1
5,publish_gold_layer,SUCCESS,2026-03-17T03:45:52.200875,2026-03-17T03:45:52.415331,0.21,Completed on attempt 1
6,generate_kpis,SUCCESS,2026-03-17T03:45:52.415634,2026-03-17T03:45:52.450974,0.04,Completed on attempt 1
7,write_pipeline_summary,SUCCESS,2026-03-17T03:45:52.451211,2026-03-17T03:45:52.451965,0.00,Completed on attempt 1


In [ ]:
print('KPI Output')
pd.read_csv(OUTPUT_PATH / 'kpi_output.csv').head()

KPI Output


,region,avg_daily_kwh_usage,avg_daily_outage_minutes,total_billed_amount
0,Central,34.901469,0.520891,7545.14
1,East,34.907781,0.511665,7118.06
2,North,35.114747,0.683378,6930.07
3,South,35.187823,0.383838,6412.30
4,West,34.965314,0.418686,8112.84


In [ ]:
with open(OUTPUT_PATH / 'pipeline_summary.json', 'r') as f:
    print(f.read())

{
  "generated_at": "2026-03-17T03:45:52.451213",
  "raw_files": 4,
  "bronze_files": 4,
  "silver_files": 4,
  "gold_files": 1,
  "output_files": 1
}


In [ ]:
def failure_simulation_task():
    raise ValueError('Simulated pipeline failure for retry testing')

try:
    run_task('failure_simulation_task', failure_simulation_task, retries=2, retry_delay=1)
except Exception as e:
    print('Expected final failure captured:', e)

failure_simulation_task: RETRYING because Simulated pipeline failure for retry testing
failure_simulation_task: RETRYING because Simulated pipeline failure for retry testing
failure_simulation_task: FAILED
Expected final failure captured: Simulated pipeline failure for retry testing
